# Verify Conv-VAE-Neo MultiView Fusion

In [ ]:
import json
import pathlib
import sys
sys.path.append("..")

import numpy as np
import torch
from exp_run_config import Config
Config.PROJECTNAME = "BerryPicker"
from sensorprocessing.multiview_data import make_multiview_dataloaders
from sensorprocessing.sp_factory import create_sp

In [ ]:
experiment = "sensorprocessing_conv_vae_neo_multiview_fusion"
run = "sp_vae_neo_multiview_fusion_128_256px"
creation_style = "exist-ok"

In [ ]:
exp = Config().get_experiment(experiment, run, creation_style=creation_style)
sp = create_sp(exp)
robot_exp = Config().get_experiment(exp["robot_exp"], exp["robot_run"])
_, validation_loader = make_multiview_dataloaders(exp, robot_exp=robot_exp)
predictions, targets, latents = [], [], []
with torch.no_grad():
    for views, target in validation_loader:
        views = [view.to(Config().runtime["device"]) for view in views]
        predictions.append(sp.enc(views).cpu())
        latents.append(sp.enc.encode_views(views).cpu())
        targets.append(target)
predictions = torch.cat(predictions)
targets = torch.cat(targets)
latents = torch.cat(latents)
metrics = {
    "rmse": float(torch.sqrt(torch.mean((predictions - targets) ** 2))),
    "mae": float(torch.mean(torch.abs(predictions - targets))),
    "latent_mean": float(latents.mean()),
    "latent_std": float(latents.std()),
    "samples": len(targets),
}
if not all(np.isfinite(value) for value in metrics.values()):
    raise FloatingPointError(f"Non-finite verification metrics: {metrics}")
print(json.dumps(metrics, indent=2))
metrics_path = pathlib.Path(exp["data_dir"], "verification_metrics.json")
with metrics_path.open("w", encoding="utf-8") as handle:
    json.dump(metrics, handle, indent=2)
    handle.write("\n")